<a href="https://colab.research.google.com/github/nathanieladitya/CODETECH-Task/blob/main/chatbott.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk
!pip install numpy

In [2]:
import numpy as np
import random
import json
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab') # Download the missing resource

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
# HDFC Bank knowledge base
hdfc_data = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["Hi", "Hello", "Hey", "Good morning", "Good afternoon"],
            "responses": ["Hello! Welcome to HDFC Bank. How can I assist you today?", "Hi there! How can I help you with your HDFC Bank account?"],
            "context_set": ""
        },
        {
            "tag": "goodbye",
            "patterns": ["Bye", "Goodbye", "See you later", "Have a good day"],
            "responses": ["Thank you for banking with HDFC. Have a great day!", "Goodbye! Visit again for your banking needs."],
            "context_set": ""
        },
        {
            "tag": "thanks",
            "patterns": ["Thanks", "Thank you", "That's helpful", "Appreciate it"],
            "responses": ["Happy to help!", "You're welcome!", "My pleasure!"],
            "context_set": ""
        },
        {
            "tag": "account_balance",
            "patterns": ["What's my balance?", "How much do I have in my account?", "Check my account balance", "Current balance"],
            "responses": ["Your current account balance is ₹12,345.67", "As of today, your HDFC account balance is ₹12,345.67"],
            "context_set": ""
        },   {
            "tag": "transactions",
            "patterns": ["Show my recent transactions", "Last 5 transactions", "Transaction history", "Recent activity"],
            "responses": ["Here are your last 5 transactions:\n1. ₹1,000.00 debit to Amazon\n2. ₹5,000.00 credit from Salary\n3. ₹500.00 debit to Uber\n4. ₹2,000.00 transfer to John\n5. ₹100.00 debit for ATM withdrawal",
                         "Your recent transactions:\n- ₹1,000 debit: Amazon\n- ₹5,000 credit: Salary"],
            "context_set": ""
   },
        {
            "tag": "fund_transfer",
            "patterns": ["Transfer money", "Send money", "Make a payment", "NEFT transfer", "IMPS transfer"],
            "responses": ["I can help you transfer funds. Please provide the recipient account number and amount.",
                          "For fund transfer, please enter the beneficiary details and amount."],
            "context_set": ""
        },
        {
            "tag": "credit_card",
            "patterns": ["Credit card payment", "Pay credit card bill", "Card due date", "Credit card limit"],
            "responses": ["Your HDFC credit card payment of ₹5,000 is due on 15th of next month. Current limit: ₹50,000",
                          "You can pay your credit card bill through NetBanking or Mobile Banking."],
            "context_set": ""
        },
        {
            "tag": "loan",
            "patterns": ["Loan options", "Personal loan", "Home loan", "Loan interest rates", "Apply for loan"],
            "responses": ["HDFC offers various loan options:\n1. Personal Loan @10.5%\n2. Home Loan @8.4%\n3. Car Loan @8.75%",
                          "You can check loan eligibility and apply through our website or mobile app."],
            "context_set": ""
        },
        {
            "tag": "customer_care",
            "patterns": ["Contact customer care", "Help number", "Customer support", "Bank phone number"],
            "responses": ["You can reach HDFC customer care at 1800 123 1234 (toll-free) or 0124-1234567 (chargeable).",
                          "For assistance, call our 24/7 customer support at 1800 123 1234."],
            "context_set": ""
        }
    ]
}



In [5]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()



In [6]:
# Preprocess data
words = []
classes = []
documents = []
ignore_words = ['?', '!', '.', ',']

for intent in hdfc_data['intents']:
    for pattern in intent['patterns']:
        # Tokenize each word
        w = word_tokenize(pattern)
        words.extend(w)
        # Add documents in the corpus
        documents.append((w, intent['tag']))
        # Add to our classes list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])


In [7]:
# Lemmatize and lower each word and remove duplicates
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))


In [8]:
# Create training data
training = []
output_empty = [0] * len(classes)

for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]

    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1

    training.append([bag, output_row])


In [9]:
# Shuffle features and convert to numpy array
random.shuffle(training)
training = np.array(training, dtype=object)

In [10]:
# Create train and test lists
train_x = list(training[:,0])
train_y = list(training[:,1])

In [11]:
# TF-IDF Vectorizer approach
def preprocess_user_input(user_input):
    # Tokenize and lemmatize the user input
    tokens = word_tokenize(user_input)
    tokens = [lemmatizer.lemmatize(word.lower()) for word in tokens]
    return ' '.join(tokens)


In [12]:
def get_response(user_input, intents_data):
    # Preprocess the user input
    processed_input = preprocess_user_input(user_input)
      # Vectorize the input and patterns
    vectorizer = TfidfVectorizer()
    # Get all patterns from intents
    all_patterns = []
    for intent in intents_data['intents']:
        all_patterns.extend(intent['patterns'])

    # Add user input temporarily to fit the vectorizer
    temp_patterns = all_patterns + [user_input]
    tfidf = vectorizer.fit_transform(temp_patterns)
  # Calculate cosine similarity
    cosine_values = cosine_similarity(tfidf[-1], tfidf[:-1])

    # Get the most similar pattern
    best_match_idx = cosine_values.argmax()
    best_match_score = cosine_values[0][best_match_idx]

    # Find which intent the best match belongs to
    current_idx = 0
    for intent in intents_data['intents']:
        if best_match_idx < current_idx + len(intent['patterns']):
   # If similarity score is high enough, return a random response from the intent
            if best_match_score > 0.6:
                return random.choice(intent['responses'])
            else:
                return "I'm sorry, I didn't understand that. Could you please rephrase?"
        current_idx += len(intent['patterns'])

    return "I'm not sure how to help with that. Please contact HDFC customer care for assistance."

In [ ]:
 # Chatbot interaction
print("HDFC Bank Chatbot: Hello! I'm your HDFC Bank assistant. Type 'quit' to exit.")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("HDFC Bank Chatbot: Thank you for banking with HDFC. Have a great day!")
        break

    response = get_response(user_input, hdfc_data)
    print("HDFC Bank Chatbot:",response)

HDFC Bank Chatbot: Hello! I'm your HDFC Bank assistant. Type 'quit' to exit.
